In [1]:
import pandas as pd
import os

DATA_PATH   = r"C:\Users\yipch\OneDrive\Desktop\olist_analytics\data"
EXPORT_PATH = r"C:\Users\yipch\OneDrive\Desktop\olist_analytics\exports"

customers   = pd.read_csv(os.path.join(DATA_PATH, "olist_customers_dataset.csv"))
orders      = pd.read_csv(os.path.join(DATA_PATH, "olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(DATA_PATH, "olist_order_items_dataset.csv"))
payments    = pd.read_csv(os.path.join(DATA_PATH, "olist_order_payments_dataset.csv"))
reviews     = pd.read_csv(os.path.join(DATA_PATH, "olist_order_reviews_dataset.csv"))
products    = pd.read_csv(os.path.join(DATA_PATH, "olist_products_dataset.csv"))
sellers     = pd.read_csv(os.path.join(DATA_PATH, "olist_sellers_dataset.csv"))
category    = pd.read_csv(os.path.join(DATA_PATH, "product_category_name_translation.csv"))

print("✅ All files loaded!")

✅ All files loaded!


In [2]:
datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print("✅ Datetime columns fixed!")
print(orders[datetime_cols].dtypes)

✅ Datetime columns fixed!
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [3]:
before = len(orders)

# Keep only delivered orders for analysis
orders_clean = orders[orders['order_status'] == 'delivered'].copy()

# Drop rows where purchase timestamp or delivery date is null
orders_clean = orders_clean.dropna(subset=['order_purchase_timestamp',
                                            'order_delivered_customer_date'])

# Add useful time columns
orders_clean['order_year_month'] = orders_clean['order_purchase_timestamp'].dt.strftime('%Y-%m')
orders_clean['order_year']       = orders_clean['order_purchase_timestamp'].dt.year
orders_clean['order_month']      = orders_clean['order_purchase_timestamp'].dt.month
orders_clean['order_dow']        = orders_clean['order_purchase_timestamp'].dt.day_name()

# Delivery time in days
orders_clean['delivery_days'] = (
    orders_clean['order_delivered_customer_date'] -
    orders_clean['order_purchase_timestamp']
).dt.days

# Delivery vs estimate (negative = early, positive = late)
orders_clean['delivery_delay_days'] = (
    orders_clean['order_delivered_customer_date'] -
    orders_clean['order_estimated_delivery_date']
).dt.days

after = len(orders_clean)
print(f"Orders: {before:,} → {after:,} (removed {before - after:,} non-delivered/null rows)")

Orders: 99,441 → 96,470 (removed 2,971 non-delivered/null rows)


In [4]:
# Fill missing category with 'unknown'
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Fill missing dimensions/weight with median
num_cols = ['product_weight_g', 'product_length_cm',
            'product_height_cm', 'product_width_cm']

for col in num_cols:
    products[col] = products[col].fillna(products[col].median())

print("✅ Products cleaned!")
print(products.isnull().sum())

✅ Products cleaned!
product_id                      0
product_category_name           0
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                0
product_length_cm               0
product_height_cm               0
product_width_cm                0
dtype: int64


In [5]:
# Only keep the columns we need
reviews_clean = reviews[['order_id', 'review_score']].copy()

# Drop duplicates — keep the first review per order
reviews_clean = reviews_clean.drop_duplicates(subset='order_id', keep='first')

print(f"✅ Reviews cleaned: {len(reviews_clean):,} rows")
print(reviews_clean['review_score'].value_counts().sort_index())

✅ Reviews cleaned: 98,673 rows
review_score
1    11353
2     3133
3     8124
4    19044
5    57019
Name: count, dtype: int64


In [6]:
# Start with clean orders
master = orders_clean.copy()

# Join customers
master = master.merge(customers[['customer_id','customer_unique_id',
                                  'customer_city','customer_state']],
                       on='customer_id', how='left')

# Join order items (aggregated per order)
items_agg = order_items.groupby('order_id').agg(
    total_items       = ('order_item_id', 'count'),
    total_revenue     = ('price', 'sum'),
    total_freight     = ('freight_value', 'sum')
).reset_index()

master = master.merge(items_agg, on='order_id', how='left')

# Join payments (aggregated per order)
pay_agg = payments.groupby('order_id').agg(
    total_payment     = ('payment_value', 'sum'),
    payment_type      = ('payment_type', lambda x: x.mode()[0])
).reset_index()

master = master.merge(pay_agg, on='order_id', how='left')

# Join reviews
master = master.merge(reviews_clean, on='order_id', how='left')

print(f"✅ Master table built: {master.shape}")
print(master.head(3))

✅ Master table built: (96470, 23)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08-07 15:27:45   
2          2018-08-08 13:50:00           2018-08-17 18:06:29   

  order_estimated_delivery_date order_year_month  order_year  ...  \
0                    2017-10-18          2017-10        2017  ...   
1                    2018

In [7]:
print("📊 Master Table Summary")
print(f"Rows:    {len(master):,}")
print(f"Columns: {master.shape[1]}")
print(f"\nDate range: {master['order_purchase_timestamp'].min().date()} "
      f"→ {master['order_purchase_timestamp'].max().date()}")
print(f"Unique customers: {master['customer_unique_id'].nunique():,}")
print(f"Total revenue:    R$ {master['total_revenue'].sum():,.2f}")
print(f"\nNull counts:\n{master.isnull().sum()[master.isnull().sum() > 0]}")

# Save
master.to_csv(os.path.join(EXPORT_PATH, "master_orders.csv"), index=False)
orders_clean.to_csv(os.path.join(EXPORT_PATH, "orders_clean.csv"), index=False)

print("\n✅ Files saved to exports/")

📊 Master Table Summary
Rows:    96,470
Columns: 23

Date range: 2016-09-15 → 2018-08-29
Unique customers: 93,350
Total revenue:    R$ 13,220,248.93

Null counts:
order_approved_at                14
order_delivered_carrier_date      1
total_payment                     1
payment_type                      1
review_score                    646
dtype: int64

✅ Files saved to exports/
